In [1]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
from datetime import date

In [2]:

def get_job_description_and_title(url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return "Job title not found.", f"Error: {e}"

    soup = BeautifulSoup(response.content, 'html.parser')

    job_description_div = soup.find('div', class_='description__text')
    job_description = job_description_div.get_text(separator='\n', strip=True) if job_description_div else "Job description not found."

    title_tag = soup.find('h1')
    job_title = title_tag.get_text(strip=True) if title_tag else "Job title not found."

    return job_title, job_description

def get_job_listing_urls(search_url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    job_urls = set()

    try:
        response = requests.get(search_url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching search URL {search_url}: {e}")
        return []

    soup = BeautifulSoup(response.content, 'html.parser')

    for a_tag in soup.find_all('a', class_='base-card__full-link'):
        if 'href' in a_tag.attrs:
            job_urls.add(a_tag['href'].split('?')[0])

    return list(job_urls)

In [3]:
if __name__ == "__main__":
    # Daftar keyword berdasarkan kategori
    keywords_cs = ["informatics", "informatics engineering", "computer science"]
    keywords_is = ["information systems", "accounting information systems", "informatics management"]
    keywords_all = keywords_cs + keywords_is

    all_data = []

    for keyword in keywords_all:
        print(f"\nFetching job listing URLs for keyword: {keyword}")
        encoded_keyword = keyword.replace(" ", "%20")
        search_page_url = f"https://www.linkedin.com/jobs/search/?keywords={encoded_keyword}&location=Indonesia&f_TPR=r86400&sortBy=R"

        urls = get_job_listing_urls(search_page_url)
        print(f"Found {len(urls)} job URLs for keyword: {keyword}")

        for i, url in enumerate(urls):
            print(f"Scraping job {i+1}/{len(urls)} for '{keyword}': {url}")
            title, description = get_job_description_and_title(url)
            all_data.append({
                "job_source": "LinkedIn",
                "keyword": keyword,
                "job_title": title,
                "job_description": description
            })
            time.sleep(1)  # Agar tidak diblokir

    # Simpan hasil gabungan
    df = pd.DataFrame(all_data)

    # Pisahkan berdasarkan kategori
    df_cs = df[df['keyword'].isin(keywords_cs)]
    df_is = df[df['keyword'].isin(keywords_is)]

    today = date.today()
    # Simpan ke file Excel
    df_cs.to_excel(f"LinkedIn_CS_{today}.xlsx", index=False)
    df_is.to_excel(f"LinkedIn_IS_{today}.xlsx", index=False)

    print("\nSelesai.")


Fetching job listing URLs for keyword: informatics
Found 5 job URLs for keyword: informatics
Scraping job 1/5 for 'informatics': https://id.linkedin.com/jobs/view/cloud-solution-architect-at-pt-computrade-technology-international-cti-group-4268703503
Scraping job 2/5 for 'informatics': https://id.linkedin.com/jobs/view/senior-data-analyst-at-traveloka-4148249224
Scraping job 3/5 for 'informatics': https://id.linkedin.com/jobs/view/goto-group-senior-data-analyst-at-goto-group-4190937397
Scraping job 4/5 for 'informatics': https://id.linkedin.com/jobs/view/marketing-technology-analyst-at-astra-financial-4268484302
Error fetching https://id.linkedin.com/jobs/view/marketing-technology-analyst-at-astra-financial-4268484302: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
Scraping job 5/5 for 'informatics': https://id.linkedin.com/jobs/view/back-end-developer-at-pt-prima-fajar-cahaya-surya-4268705198

F